#  Notebook 13 - Run All Clustering Models

Produces all clustering result files from the imputed characteristics.

## Reads from
```
data/raw/
    Imputed__characteristics_winsorized99.csv
    Imputed_characteristics_with_MktCap_SICCD_(inner_join).csv
    FF3.csv
```

## Writes to
```
data/grid/
    vanilla/
        clustering_results_no_inertia_K_10.csv
        clustering_results_no_inertia_K_50.csv
    inertia/
        clustering_results_K_10.csv
        clustering_results_K_50.csv
    temporal_loss/
        clustering_results_K_2_lambda_1e-09.csv
        ... (27 K values x 19 lambda values = 513 files)
        clustering_results_K_100_lambda_1000000000.csv
```

## Output columns (all models)
`year_month, permno, cluster, return, MktCap, SICCD, RF, excess_return`

## Models

| Cell | Model | Description |
|------|-------|-------------|
| 4 | Vanilla | Fresh k-means++ each month - no temporal memory |
| 5 | Inertia | Previous month centroids as warm-start |
| 6-7 | Dynamic | Temporal drift penalty: minimises sum||x-mu||^2 + lambda*||mu-mu_prev||^2 * N/K |

## Runtime
- Vanilla + Inertia: ~5 minutes
- Dynamic full grid (513 runs): several hours
- Resume-safe: already-computed files are skipped automatically

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from IPython.display import display
import warnings
warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────
# Notebook lives in:  Replication Package/notebooks/
# Reads from:         Replication Package/data/raw/
# Writes to:          Replication Package/data/grid/

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

RAW_DIR  = os.path.join(REPO_ROOT, "data", "raw")
GRID_DIR = os.path.join(REPO_ROOT, "data", "grid")

CHARS_FILE  =  "/ssd1/songjiangliu/shared/asset_clustering/Data/Imputed__characteristics_winsorized99.csv"
MKTCAP_FILE = "/ssd1/songjiangliu/shared/asset_clustering/Data/Imputed_characteristics_with_MktCap_SICCD_(inner_join).csv"
FF3_FILE    = os.path.join(RAW_DIR, "FF3.csv")

# Output directories
os.makedirs(os.path.join(GRID_DIR, "vanilla"),       exist_ok=True)
os.makedirs(os.path.join(GRID_DIR, "inertia"),       exist_ok=True)
os.makedirs(os.path.join(GRID_DIR, "temporal_loss"), exist_ok=True)

# ── Grid ───────────────────────────────────────────────────────────
K_VANILLA  = [10, 50]
K_INERTIA  = [10, 50]
K_TEMPORAL = [2,3,4,5,6,7,8,9,10,15,20,25,30,35,40,45,50,
              55,60,65,70,75,80,85,90,95,100]
LAMBDA_VALUES = [10**i for i in range(-9, 10)]   # 1e-9 ... 1e9

FEATURES = [
    'A2ME','AC','AT','ATO','B2M','BETA_d','BETA_m','C2A','CF2B','CF2P',
    'CTO','D2A','D2P','DPI2A','E2P','FC2Y','HIGH52','INV','IdioVol','LEV',
    'ME','NI','NOA','OA','OL','OP','PCM','PM','PROF','Q',
    'R12_2','R12_7','R2_1','R36_13','R60_13','RNA','ROA','ROE','RVAR',
    'S2P','SGA2S','SPREAD','SUV','TURN','VAR',
]

print("Paths configured.")
print(f"  Repo root   : {REPO_ROOT}")
print(f"  Raw data    : {RAW_DIR}")
print(f"  Grid output : {GRID_DIR}")
print(f"  Chars file  : {os.path.exists(CHARS_FILE)}")
print(f"  MktCap file : {os.path.exists(MKTCAP_FILE)}")
print(f"  FF3 file    : {os.path.exists(FF3_FILE)}")
print(f"  K temporal  : {K_TEMPORAL}")
print(f"  Lambda grid : {len(LAMBDA_VALUES)} values ({LAMBDA_VALUES[0]} ... {LAMBDA_VALUES[-1]})")

Paths configured.
  Repo root   : /ssd1/songjiangliu/shared/asset_clustering/Replication Package
  Raw data    : /ssd1/songjiangliu/shared/asset_clustering/Replication Package/data/raw
  Grid output : /ssd1/songjiangliu/shared/asset_clustering/Replication Package/data/grid
  Chars file  : True
  MktCap file : True
  FF3 file    : True
  K temporal  : [2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
  Lambda grid : 19 values (1e-09 ... 1000000000)


In [2]:
# ── Load and prepare base data ─────────────────────────────────────
print("Loading characteristics file...")
df = pd.read_csv(CHARS_FILE, low_memory=False)

# Parse date — format is '1977-01-31'
df['date'] = pd.to_datetime(df['date'], infer_datetime_format=True, errors='coerce')
df = df.dropna(subset=['date'])
df = df.sort_values('date').reset_index(drop=True)
df['year_month'] = df['date'].dt.strftime('%Y-%m')

# Confirm features are present
feat_cols = [f for f in FEATURES if f in df.columns]
missing   = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"WARNING: {len(missing)} features missing: {missing}")
print(f"Data loaded   : {df.shape}")
print(f"Date range    : {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Features      : {len(feat_cols)} / {len(FEATURES)}")

# ── Load auxiliary data (MktCap, SICCD, RF) ────────────────────────
print("\nLoading MktCap/SICCD file (may take a minute)...")
mktcap_df = pd.read_csv(MKTCAP_FILE,
                         usecols=['date','permno','SICCD','MktCap'],
                         low_memory=False)
mktcap_df['date'] = pd.to_datetime(mktcap_df['date'],
                                    infer_datetime_format=True, errors='coerce')
mktcap_df['year_month'] = mktcap_df['date'].dt.strftime('%Y-%m')
mktcap_df.drop(columns=['date'], inplace=True)
mktcap_df = mktcap_df.drop_duplicates(subset=['year_month','permno'])
print(f"MktCap data   : {mktcap_df.shape}")

print("Loading FF3 file...")
ff3_df = pd.read_csv(FF3_FILE)
ff3_df['year_month'] = pd.to_datetime(
    ff3_df['year_month'], infer_datetime_format=True).dt.strftime('%Y-%m')
ff3_df = ff3_df[['year_month','RF']].drop_duplicates(subset=['year_month'])
print(f"FF3 data      : {ff3_df.shape}")
print("\nAll data loaded.")

Loading characteristics file...
Data loaded   : (286000, 50)
Date range    : 1977-01-31 -> 1982-10-31
Features      : 45 / 45

Loading MktCap/SICCD file (may take a minute)...
MktCap data   : (2736114, 4)
Loading FF3 file...
FF3 data      : (1182, 2)

All data loaded.


In [3]:
# ── Shared helper: enrich and save clustering results ─────────────
def enrich_and_save(clustered_df, fname):
    out = clustered_df.copy()
    out['year_month'] = pd.to_datetime(
        out['year_month'], errors='coerce').dt.strftime('%Y-%m')

    # Merge return
    df_ret = df[['year_month','permno','return']].drop_duplicates(
                subset=['year_month','permno'])
    out = out.merge(df_ret, on=['year_month','permno'], how='left')

    # Merge MktCap and SICCD
    out = out.merge(mktcap_df, on=['year_month','permno'], how='left')

    # Merge RF and compute excess return
    out = out.merge(ff3_df, on='year_month', how='left')
    out['excess_return'] = out['return'] - out['RF'] / 100

    out.to_csv(fname, index=False)
    print(f"  Saved -> {fname.replace(REPO_ROOT, '.')}")
    return out

print("Helper function defined.")

Helper function defined.


In [4]:
# ══════════════════════════════════════════════════════════════════
# MODEL 1: Vanilla K-Means
# Fresh k-means++ initialisation every month — no temporal memory
# ══════════════════════════════════════════════════════════════════
print("=" * 60)
print("MODEL 1: Vanilla K-Means")
print("=" * 60)

unique_months = sorted(df['date'].unique())

for k in K_VANILLA:
    print(f"\n  K={k}...")
    assignments = []

    for month in unique_months:
        monthly = df[df['date'] == month]
        X       = monthly[feat_cols].values
        kmeans  = KMeans(n_clusters=k, init='k-means++',
                         n_init=1, random_state=68)
        kmeans.fit(X)
        tmp = monthly[['permno','year_month']].copy()
        tmp['cluster'] = kmeans.labels_
        assignments.append(tmp)

    results_df = pd.concat(assignments, ignore_index=True)
    results_df = results_df[['year_month','permno','cluster']]

    fname = os.path.join(GRID_DIR, "vanilla",
                         f"clustering_results_no_inertia_K_{k}.csv")
    enrich_and_save(results_df, fname)

print("\nVanilla K-Means complete.")

MODEL 1: Vanilla K-Means

  K=10...
  Saved -> ./data/grid/vanilla/clustering_results_no_inertia_K_10.csv

  K=50...
  Saved -> ./data/grid/vanilla/clustering_results_no_inertia_K_50.csv

Vanilla K-Means complete.


In [5]:
# ══════════════════════════════════════════════════════════════════
# MODEL 2: Inertia K-Means
# Warm-start: previous month's centroids used as initialisation
# ══════════════════════════════════════════════════════════════════
print("=" * 60)
print("MODEL 2: Inertia K-Means (centroid warm-start)")
print("=" * 60)

unique_months = sorted(df['date'].unique())

for k in K_INERTIA:
    print(f"\n  K={k}...")
    prev_centroids = None
    assignments    = []

    for month in unique_months:
        monthly = df[df['date'] == month]
        X       = monthly[feat_cols].values
        init    = prev_centroids if prev_centroids is not None else 'k-means++'
        kmeans  = KMeans(n_clusters=k, init=init,
                         n_init=1, random_state=68)
        kmeans.fit(X)
        prev_centroids = kmeans.cluster_centers_
        tmp = monthly[['permno','year_month']].copy()
        tmp['cluster'] = kmeans.labels_
        assignments.append(tmp)

    results_df = pd.concat(assignments, ignore_index=True)
    results_df = results_df[['year_month','permno','cluster']]

    fname = os.path.join(GRID_DIR, "inertia",
                         f"clustering_results_K_{k}.csv")
    enrich_and_save(results_df, fname)

print("\nInertia K-Means complete.")

MODEL 2: Inertia K-Means (centroid warm-start)

  K=10...
  Saved -> ./data/grid/inertia/clustering_results_K_10.csv

  K=50...
  Saved -> ./data/grid/inertia/clustering_results_K_50.csv

Inertia K-Means complete.


In [6]:
# ══════════════════════════════════════════════════════════════════
# MODEL 3: Dynamic K-Means with Temporal Drift Penalty
#
# Minimises: L = sum_i ||x_i - mu_{c_i}||^2
#              + lambda * ||mu - mu_prev||^2 * N/K
#
# The N/K factor normalises for sample size vs number of clusters,
# matching the paper's implementation exactly.
# ══════════════════════════════════════════════════════════════════

def dynamic_kmeans_with_drift(X, prev_centroids, n_clusters,
                               lambda_drift, max_iter=5, tol=1e-4):
    # First month: no previous centroids — run standard k-means++
    if prev_centroids is None or np.all(prev_centroids == 0):
        km     = KMeans(n_clusters=n_clusters, init='k-means++',
                        n_init=10, random_state=0)
        labels = km.fit_predict(X)
        cents  = km.cluster_centers_
        return ((X - cents[labels]) ** 2).sum(), labels, cents

    cents = prev_centroids.copy()
    for _ in range(max_iter):
        # E-step: assign to nearest centroid
        dists  = np.linalg.norm(
            X[:, None, :] - cents[None, :, :], axis=2) ** 2
        labels = np.argmin(dists, axis=1)

        # M-step: update with drift regularisation
        new_cents = np.zeros_like(cents)
        for j in range(n_clusters):
            members = X[labels == j]
            if members.size:
                new_cents[j] = (members.sum(axis=0)
                                + lambda_drift * prev_centroids[j])                                / (len(members) + lambda_drift)
            else:
                new_cents[j] = prev_centroids[j]

        if np.linalg.norm(new_cents - cents) < tol:
            break
        cents = new_cents

    feat_loss  = ((X - cents[labels]) ** 2).sum()
    drift_loss = ((cents - prev_centroids) ** 2).sum()                  * X.shape[0] / n_clusters
    return feat_loss + lambda_drift * drift_loss, labels, cents

print("dynamic_kmeans_with_drift defined.")

dynamic_kmeans_with_drift defined.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# RUN Model 3: full K x lambda grid
# Resume-safe: skips files that already exist in data/grid/
# ══════════════════════════════════════════════════════════════════
print("=" * 60)
print("MODEL 3: Dynamic K-Means — full K x lambda grid")
print(f"  {len(K_TEMPORAL)} K values x {len(LAMBDA_VALUES)} lambda values = "
      f"{len(K_TEMPORAL)*len(LAMBDA_VALUES)} total runs")
print("=" * 60)

unique_months = sorted(df['date'].unique())
n_total  = len(K_TEMPORAL) * len(LAMBDA_VALUES)
n_done   = 0
n_skip   = 0

for k in K_TEMPORAL:
    # Pre-group monthly data once per K (reused across all lambdas)
    monthly_data = {m: df[df['date'] == m] for m in unique_months}

    for lam in LAMBDA_VALUES:
        fname = os.path.join(GRID_DIR, "temporal_loss",
                             f"clustering_results_K_{k}_lambda_{lam}.csv")

        # Skip if already computed
        if os.path.exists(fname):
            n_done += 1
            n_skip += 1
            continue

        prev_centroids = np.zeros((k, len(feat_cols)))
        assignments    = []

        for month in unique_months:
            monthly = monthly_data[month]
            X       = monthly[feat_cols].values
            _, labels, prev_centroids = dynamic_kmeans_with_drift(
                X, prev_centroids, n_clusters=k, lambda_drift=lam
            )
            tmp = monthly[['permno','year_month']].copy()
            tmp['cluster'] = labels
            assignments.append(tmp)

        results_df = pd.concat(assignments, ignore_index=True)
        results_df = results_df[['year_month','permno','cluster']]
        enrich_and_save(results_df, fname)

        n_done += 1
        if n_done % 10 == 0:
            print(f"  [{n_done:4d}/{n_total}]  K={k:3d}  lambda={lam}"
                  f"  (skipped={n_skip})")

print(f"\nModel 3 complete.")
print(f"  Total         : {n_done}")
print(f"  Skipped       : {n_skip}")
print(f"  Newly computed: {n_done - n_skip}")

MODEL 3: Dynamic K-Means — full K x lambda grid
  27 K values x 19 lambda values = 513 total runs
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_1e-09.csv
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_1e-08.csv
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_1e-07.csv
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_1e-06.csv
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_1e-05.csv
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_0.0001.csv
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_0.001.csv
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_0.01.csv
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_0.1.csv
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_1.csv
  [  10/513]  K=  2  lambda=1  (skipped=0)
  Saved -> ./data/grid/temporal_loss/clustering_results_K_2_lambda_10.csv
  Saved -> ./data/gri

In [ ]:
# ══════════════════════════════════════════════════════════════════
# VERIFY: check all expected output files exist
# ══════════════════════════════════════════════════════════════════
missing = []

# Vanilla
for k in K_VANILLA:
    p = os.path.join(GRID_DIR, "vanilla",
                     f"clustering_results_no_inertia_K_{k}.csv")
    if not os.path.exists(p):
        missing.append(p.replace(REPO_ROOT, '.'))

# Inertia
for k in K_INERTIA:
    p = os.path.join(GRID_DIR, "inertia",
                     f"clustering_results_K_{k}.csv")
    if not os.path.exists(p):
        missing.append(p.replace(REPO_ROOT, '.'))

# Temporal loss
for k in K_TEMPORAL:
    for lam in LAMBDA_VALUES:
        p = os.path.join(GRID_DIR, "temporal_loss",
                         f"clustering_results_K_{k}_lambda_{lam}.csv")
        if not os.path.exists(p):
            missing.append(p.replace(REPO_ROOT, '.'))

total_expected = (len(K_VANILLA) + len(K_INERTIA)
                  + len(K_TEMPORAL) * len(LAMBDA_VALUES))
print(f"Expected files : {total_expected}")
print(f"Missing files  : {len(missing)}")
if missing:
    print("\nFirst 10 missing:")
    for p in missing[:10]:
        print(f"  {p}")
else:
    print("All files present - ready to run replication notebooks.")